In [10]:
from qdrant_client import QdrantClient
from qdrant_client import models


client = QdrantClient(host="localhost", port=6333)
client.create_collection(
    collection_name="test_collection",
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE
    )
)

points = [
    models.PointStruct(
        id=1,
        vector=[0.1, 0.2, 0.3, 0.4],  # 4D vector
        payload={"category": "example"}  # Metadata (optional)
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.3, 0.4, 0.5],
        payload={"category": "demo"}
    )
]

client.upsert(
    collection_name = "test_collection",
    points=points
)

collection_info = client.get_collection(collection_name="test_collection")

print(f"Collection Name: {collection_info}")

Collection Name: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=2 segments_count=4 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=4, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_thread

In [11]:
from qdrant_client import QdrantClient
from qdrant_client import models

client = QdrantClient(host="localhost", port=6333)
collection_name = "test_collection"

query_vector = [0.08, 0.14, 0.33, 0.28]

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=1  # Return the top 1 most similar vector
)

print("Search results:", search_results)

Search results: points=[ScoredPoint(id=1, version=1, score=0.97642946, payload={'category': 'example'}, vector=None, shard_key=None, order_value=None)]


In [12]:
from src.pdf_loader import process_knowledge_base

chunks = process_knowledge_base("test_directory")
chunks

Erreur : Le dossier 'test_directory' n'existe pas.
Veuillez créer un dossier 'data/raw' et y glisser quelques PDF.


[]

In [13]:
from src.pdf_loader import process_knowledge_base
from src.embedding import search_in_qdrant, build_vector_store, store_embbedings_in_qdrant

chunks = process_knowledge_base("data/raw")

vector_store = build_vector_store(chunks)
store_embbedings_in_qdrant(chunks, vector_store)

Traitement en cours : Charte-alimentation-responsable-AU-CA-10.09.2024.pdf
   -> 8 chunks générés pour Charte-alimentation-responsable-AU-CA-10.09.2024.pdf
Traitement en cours : Charte-pour-lusage-du-SI-et-des-services-numériques-adoptée-en-CA-le-26-septembre-2017.pdf
   -> 14 chunks générés pour Charte-pour-lusage-du-SI-et-des-services-numériques-adoptée-en-CA-le-26-septembre-2017.pdf
Traitement en cours : Charte-pour-l®galit®-entre-femmes-et-hommes_UAPV.pdf
   -> 0 chunks générés pour Charte-pour-l®galit®-entre-femmes-et-hommes_UAPV.pdf
Traitement en cours : Charte_des_Associations_2015-UAPV.pdf
   -> 15 chunks générés pour Charte_des_Associations_2015-UAPV.pdf
Traitement en cours : Guide-des-modalites-de-controle-des-connaissances.pdf
   -> 34 chunks générés pour Guide-des-modalites-de-controle-des-connaissances.pdf
Traitement en cours : RI-AU-mars-2025.pdf
   -> 86 chunks générés pour RI-AU-mars-2025.pdf
Traitement en cours : STATUTS-Avignon-Universite-6-fevrier-2025.pdf
   -> 78 c

In [14]:
# tests
from src.embedding import search_in_qdrant

resutls = search_in_qdrant("Qui doit être le président d’une association étudiante ?", top_k=3)
resutls

points=[ScoredPoint(id=24, version=2, score=0.8766326, payload={'producer': 'PDFCreator 2.1.2.0', 'creator': 'PDFCreator 2.1.2.0', 'creationdate': '2015-11-02T10:35:54+01:00', 'source': 'data/raw/Charte_des_Associations_2015-UAPV.pdf', 'file_path': 'data/raw/Charte_des_Associations_2015-UAPV.pdf', 'total_pages': 5, 'format': 'PDF 1.5', 'title': 'Modifications Charte des Associations 2015-1', 'author': 'Piaumier Alexandra', 'subject': '', 'keywords': '', 'moddate': '2015-11-02T10:35:54+01:00', 'trapped': '', 'modDate': "D:20151102103554+01'00'", 'creationDate': "D:20151102103554+01'00'", 'page': 0, 'source_file': 'Charte_des_Associations_2015-UAPV.pdf', 'text': 'Est association étudiante toute association reconnue comme telle par l’Université d’Avignon \net des Pays de Vaucluse (ci-après dénommée l’Université) au regard des critères cumulatifs \ndéfinis ci-après, et signataire de la présente Charte. \n \n1) Structure : \na) Le bureau de l’association doit être composé, au moins en major

[{'id': 24,
  'score': 0.8766326,
  'metadata': {'producer': 'PDFCreator 2.1.2.0',
   'creator': 'PDFCreator 2.1.2.0',
   'creationdate': '2015-11-02T10:35:54+01:00',
   'source': 'data/raw/Charte_des_Associations_2015-UAPV.pdf',
   'file_path': 'data/raw/Charte_des_Associations_2015-UAPV.pdf',
   'total_pages': 5,
   'format': 'PDF 1.5',
   'title': 'Modifications Charte des Associations 2015-1',
   'author': 'Piaumier Alexandra',
   'subject': '',
   'keywords': '',
   'moddate': '2015-11-02T10:35:54+01:00',
   'trapped': '',
   'modDate': "D:20151102103554+01'00'",
   'creationDate': "D:20151102103554+01'00'",
   'page': 0,
   'source_file': 'Charte_des_Associations_2015-UAPV.pdf'},
  'text': 'Est association étudiante toute association reconnue comme telle par l’Université d’Avignon \net des Pays de Vaucluse (ci-après dénommée l’Université) au regard des critères cumulatifs \ndéfinis ci-après, et signataire de la présente Charte. \n \n1) Structure : \na) Le bureau de l’association 

In [15]:
results = search_in_qdrant("Quelle proportion du bureau doit être composée d’étudiants ?", top_k=3)
resutls

points=[ScoredPoint(id=212, version=2, score=0.85514987, payload={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-02-13T12:34:41+01:00', 'source': 'data/raw/STATUTS-Avignon-Universite-6-fevrier-2025.pdf', 'file_path': 'data/raw/STATUTS-Avignon-Universite-6-fevrier-2025.pdf', 'total_pages': 20, 'format': 'PDF 1.7', 'title': '', 'author': 'Mission Culture et Communication', 'subject': '', 'keywords': '', 'moddate': '2025-02-13T12:34:41+01:00', 'trapped': '', 'modDate': "D:20250213123441+01'00'", 'creationDate': "D:20250213123441+01'00'", 'page': 13, 'source_file': 'STATUTS-Avignon-Universite-6-fevrier-2025.pdf', 'text': 'répartition, tel que défini par le conseil d’administration ;  \n2°) Les règles relatives aux examens ;  \n3°) Les règles d’évaluation des enseignements ;  \n4°) Des mesures recherchant la réussite du plus grand nombre d’étudiants ; \n5°) Les mesures de nature à permettre la mise en œuvre de l’orientation des étudiants et \nde

[{'id': 24,
  'score': 0.8766326,
  'metadata': {'producer': 'PDFCreator 2.1.2.0',
   'creator': 'PDFCreator 2.1.2.0',
   'creationdate': '2015-11-02T10:35:54+01:00',
   'source': 'data/raw/Charte_des_Associations_2015-UAPV.pdf',
   'file_path': 'data/raw/Charte_des_Associations_2015-UAPV.pdf',
   'total_pages': 5,
   'format': 'PDF 1.5',
   'title': 'Modifications Charte des Associations 2015-1',
   'author': 'Piaumier Alexandra',
   'subject': '',
   'keywords': '',
   'moddate': '2015-11-02T10:35:54+01:00',
   'trapped': '',
   'modDate': "D:20151102103554+01'00'",
   'creationDate': "D:20151102103554+01'00'",
   'page': 0,
   'source_file': 'Charte_des_Associations_2015-UAPV.pdf'},
  'text': 'Est association étudiante toute association reconnue comme telle par l’Université d’Avignon \net des Pays de Vaucluse (ci-après dénommée l’Université) au regard des critères cumulatifs \ndéfinis ci-après, et signataire de la présente Charte. \n \n1) Structure : \na) Le bureau de l’association 

In [16]:
results = search_in_qdrant("Qui coordonne le conseil des associations ?", top_k=3)
results

points=[ScoredPoint(id=103, version=2, score=0.85402304, payload={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-03-24T16:31:55+01:00', 'source': 'data/raw/RI-AU-mars-2025.pdf', 'file_path': 'data/raw/RI-AU-mars-2025.pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': '', 'author': 'Jessica CRISTIANINI', 'subject': '', 'keywords': '', 'moddate': '2025-03-24T16:31:55+01:00', 'trapped': '', 'modDate': "D:20250324163155+01'00'", 'creationDate': "D:20250324163155+01'00'", 'page': 9, 'source_file': 'RI-AU-mars-2025.pdf', 'text': "le Conseil d'administration de l'Université, après avis de la Commission de la \nformation et de la vie universitaire. À cette fin, elles doivent préalablement \ncommuniquer au/à la Président.e de l’Université les statuts de leur \nassociation et les coordonnées des responsables associatifs (membres du \nbureau de l’association) ainsi que toute modification ultérieure de ces \ndocuments, un rapport d’activité annuel 

[{'id': 103,
  'score': 0.85402304,
  'metadata': {'producer': 'Microsoft® Word 2019',
   'creator': 'Microsoft® Word 2019',
   'creationdate': '2025-03-24T16:31:55+01:00',
   'source': 'data/raw/RI-AU-mars-2025.pdf',
   'file_path': 'data/raw/RI-AU-mars-2025.pdf',
   'total_pages': 25,
   'format': 'PDF 1.7',
   'title': '',
   'author': 'Jessica CRISTIANINI',
   'subject': '',
   'keywords': '',
   'moddate': '2025-03-24T16:31:55+01:00',
   'trapped': '',
   'modDate': "D:20250324163155+01'00'",
   'creationDate': "D:20250324163155+01'00'",
   'page': 9,
   'source_file': 'RI-AU-mars-2025.pdf'},
  'text': "le Conseil d'administration de l'Université, après avis de la Commission de la \nformation et de la vie universitaire. À cette fin, elles doivent préalablement \ncommuniquer au/à la Président.e de l’Université les statuts de leur \nassociation et les coordonnées des responsables associatifs (membres du \nbureau de l’association) ainsi que toute modification ultérieure de ces \ndocu

In [17]:
from src.rag import build_context

context, sources = build_context(results)
context

"[1] (RI-AU-mars-2025.pdf, p.10)\nle Conseil d'administration de l'Université, après avis de la Commission de la \nformation et de la vie universitaire. À cette fin, elles doivent préalablement \ncommuniquer au/à la Président.e de l’Université les statuts de leur \nassociation et les coordonnées des responsables associatifs (membres du \nbureau de l’association) ainsi que toute modification ultérieure de ces \ndocuments, un rapport d’activité annuel et un compte-rendu des assemblées \ngénérales. \n \nL’Université peut mettre gracieusement des locaux à la disposition des \ndiverses associations des membres de la communauté universitaire, par \ndécision de son/sa Président.e. Le Président peut retirer à tout moment la \njouissance de ces locaux. \nChaque mise à disposition de locaux fait l’objet d’un acte unilatéral ou d’une \nconvention précisant les droits et obligations de leurs bénéficiaires. \nLes associations bénéficiant de cet avantage en nature sont tenues de \ncommuniquer un bil

In [18]:
sources

[{'number': 1,
  'source_file': 'RI-AU-mars-2025.pdf',
  'page': 10,
  'score': 0.85402304},
 {'number': 2,
  'source_file': 'STATUTS-Avignon-Universite-6-fevrier-2025.pdf',
  'page': 10,
  'score': 0.8503983},
 {'number': 3,
  'source_file': 'Charte_des_Associations_2015-UAPV.pdf',
  'page': 2,
  'score': 0.84755933}]

In [19]:
from src.rag import answer
result = answer("Qui doit être le président d’une association étudiante ?", 3)
result

points=[ScoredPoint(id=24, version=2, score=0.8766326, payload={'producer': 'PDFCreator 2.1.2.0', 'creator': 'PDFCreator 2.1.2.0', 'creationdate': '2015-11-02T10:35:54+01:00', 'source': 'data/raw/Charte_des_Associations_2015-UAPV.pdf', 'file_path': 'data/raw/Charte_des_Associations_2015-UAPV.pdf', 'total_pages': 5, 'format': 'PDF 1.5', 'title': 'Modifications Charte des Associations 2015-1', 'author': 'Piaumier Alexandra', 'subject': '', 'keywords': '', 'moddate': '2015-11-02T10:35:54+01:00', 'trapped': '', 'modDate': "D:20151102103554+01'00'", 'creationDate': "D:20151102103554+01'00'", 'page': 0, 'source_file': 'Charte_des_Associations_2015-UAPV.pdf', 'text': 'Est association étudiante toute association reconnue comme telle par l’Université d’Avignon \net des Pays de Vaucluse (ci-après dénommée l’Université) au regard des critères cumulatifs \ndéfinis ci-après, et signataire de la présente Charte. \n \n1) Structure : \na) Le bureau de l’association doit être composé, au moins en major

{'answer': 'Le président (ou la présidente) d’une association étudiante doit être un étudiant inscrit à l’Université d’Avignon et des Pays de Vaucluse.\u202f[1]',
 'sources': [{'number': 1,
   'source_file': 'Charte_des_Associations_2015-UAPV.pdf',
   'page': 1,
   'score': 0.8766326},
  {'number': 2,
   'source_file': 'Charte_des_Associations_2015-UAPV.pdf',
   'page': 2,
   'score': 0.87493265},
  {'number': 3,
   'source_file': 'RI-AU-mars-2025.pdf',
   'page': 10,
   'score': 0.8703679}]}

In [39]:
from fastembed import SparseTextEmbedding

embedding = SparseTextEmbedding(model_name="Qdrant/bm25",language="french")

test = embedding.embed("L'article 14 précise les conditions d'inscription")

In [40]:
test_2 = list(test)
test_2[0]

SparseEmbedding(values=array([1.66973021, 1.66973021, 1.66973021, 1.66973021, 1.66973021]), indices=array([1257441684, 1749031367, 1582480262, 1240220851,  626305400]))

In [41]:
test_2[0].indices

array([1257441684, 1749031367, 1582480262, 1240220851,  626305400])

## Reranker - Palier 1 : découvrir le cross-encoder

Le cross-encoder lit la paire **(question, texte)** ensemble et renvoie **un score de pertinence**.
Modèle : `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (multilingue, léger). Pas de préfixe `query:` / `passage:` (c'est propre à e5).

In [ ]:
import time

import torch
from sentence_transformers import CrossEncoder

# Same device choice as the e5 embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"

start = time.perf_counter()
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", device=device)
print(f"Reranker chargé sur {device} en {time.perf_counter() - start:.1f} s")

In [ ]:
# 1 question, 3 texts: one answers, one is on the same topic without answering, one is off-topic
question = "Qui peut présider une association étudiante ?"
texts = [
    "Le (la) Président(e) de l'association doit être un étudiant inscrit à l'Université d'Avignon et des Pays de Vaucluse.",  # answers
    "Les associations étudiantes peuvent demander la mise à disposition d'un local à l'Université.",                         # same topic
    "Le restaurant universitaire propose 60% de produits durables, dont au moins 20% de produits bio.",                       # off-topic
]

# predict() takes a list of (question, text) PAIRS and returns one score per pair
pairs = [(question, text) for text in texts]

start = time.perf_counter()
scores = reranker.predict(pairs)
print(f"predict : {time.perf_counter() - start:.2f} s\n")

for score, text in zip(scores, texts):
    print(f"{score:7.2f}  {text[:70]}")

In [ ]:
# rank() does the same thing, but sorts for us: it returns the indices of the texts, best first
ranking = reranker.rank(question, texts)

for item in ranking:
    print(f"{item['score']:7.2f}  texte n°{item['corpus_id']}  {texts[item['corpus_id']][:60]}")

In [ ]:
# The scores are logits (about -10 to +10), not probabilities.
# The sigmoid 1 / (1 + e^-x) maps them between 0 and 1 (useful later for a relevance threshold)
import numpy as np

probabilities = 1 / (1 + np.exp(-scores))

for score, proba, text in zip(scores, probabilities, texts):
    print(f"logit {score:6.2f}  ->  sigmoïde {proba:.3f}  {text[:50]}")